# Recurrent neural networks (RNNs)

_Deep Learning_

**Read a sequence one step at a time, carrying a memory of what came before.**

Some data is a sequence: words in a sentence, prices over days, sounds in speech. Order matters.

     A recurrent neural network (RNN) reads it one step at a time and keeps a memory of what it has seen.

---

This notebook is a practice scaffold for the **Recurrent neural networks (RNNs)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Create the recurrent layer

`nn.RNN` is a recurrent layer: it reads a sequence one step at a time and updates a hidden state that carries forward. Here `input_size=3` features come in at each step and the hidden state is a 5-dimensional vector. `batch_first=True` means we'll feed tensors shaped `(batch, time, features)`.

In [ ]:
import torch
import torch.nn as nn

rnn = nn.RNN(input_size=3, hidden_size=5, batch_first=True)

### Step 2 — Run a batch of sequences through it

We feed in 2 sequences, each 7 time-steps long with 3 features per step. The layer returns two things: `output`, the hidden state at *every* step, and `h_n`, the final hidden state only. Think of `output` as the running memory snapshot at each moment, and `h_n` as the memory after the whole sequence has been read.

In [ ]:
# batch of 2 sequences, each 7 time-steps long, 3 features per step
x = torch.randn(2, 7, 3)
output, h_n = rnn(x)        # output: every step's hidden state; h_n: the last one

### Step 3 — Inspect the shapes and confirm the link

`output` has shape `(2, 7, 5)` — a 5-dim hidden vector for each of the 7 steps in both sequences. `h_n` has shape `(1, 2, 5)` — just the final step's memory. Because the last step's hidden state *is* the carried memory, `output[:, -1, :]` should equal `h_n[0]`; the `allclose` check confirms it.

In [ ]:
print("output:", tuple(output.shape))   # (2, 7, 5) -- a hidden vector per step
print("final h:", tuple(h_n.shape))     # (1, 2, 5) -- the carried memory
# the last step's output equals the final hidden state:
print("match:", torch.allclose(output[:, -1, :], h_n[0]))

## Visualize the data & results

_Scanning a real digit image row by row, how far back can a plain recurrent step remember an early row?_

### Step 1 — Model how memory decays each step

A plain recurrent step multiplies its state by roughly the same factor every time-step. If that factor is below 1 (here ~0.7), an early signal shrinks geometrically and is soon forgotten. An LSTM's forget gate can stay near 1 (here ~0.983), so the same signal barely fades. We compute both curves over 20 time-steps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t = np.arange(0, 20)                      # timesteps (e.g. rows of an image, then more)

rnn = 0.7 ** t                            # plain RNN: state shrinks ~0.7 each step
lstm = 0.983 ** t                         # LSTM: forget gate ~1 holds the signal

### Step 2 — Plot how far back each can remember

Plotting the two curves makes the contrast vivid: the plain RNN's signal magnitude drops toward zero within a dozen steps (the vanishing-memory problem), while the LSTM line stays high. This is the intuition for why LSTMs handle long-range dependencies that plain RNNs lose.

In [ ]:
plt.plot(t, rnn, color="#ff7b72", label="plain RNN (decays)")
plt.plot(t, lstm, color="#7ee787", label="LSTM (holds)")
plt.title("Memory of an early signal across timesteps (RNN vs LSTM)")
plt.xlabel("timestep"); plt.ylabel("signal magnitude")
plt.legend()
plt.show()